# AI-103 Module 1: one grounded agent, all four decisions

**Lesson 1: Choose Foundry services, models, and integration approach.**

Module 1 is four decisions: which **model**, which **service**, which **retrieval** method, and which **memory + tools + knowledge** integration. Rather than four toy scripts, this notebook builds the **one** artifact those decisions produce: a small Contoso support agent, the way AI-103 actually ships it.

We use the **Microsoft Agent Framework** on **Microsoft Foundry** (Foundry is Microsoft's platform for building and running AI apps and agents). This is the current generally available (GA) path. The classic `create_agent` / thread / run agent application programming interface (API) is deprecated and retires **March 31, 2027**, so we deliberately do not use it.

Each learning objective (LO) maps to one ingredient of the agent, exactly like the slide "an agent is memory + tools + knowledge, with guardrails":

| LO | Decision | The ingredient in this notebook |
|---|---|---|
| **LO1** | choose a **model** | the agent reasons with your `FOUNDRY_MODEL` deployment |
| **LO2** | choose a **service** | the Foundry **Agent Service** via `FoundryChatClient` |
| **LO3** | choose **retrieval** | grounding on a file with the **file-search** tool |
| **LO4** | **memory + tools + knowledge** | a session (memory), a custom tool, the knowledge source, and an approval gate (the guardrail) |

**Before you run:** `uv sync`, then `az login`, then copy `.env.example` to `.env` and fill in your Foundry values. Run the cells **top to bottom**.

## Setup: imports, environment, and the knowledge base

We import the Agent Framework, authenticate **keyless** with the Azure command-line interface (CLI) sign-in (no application programming interface keys in code, the AI-103 security posture), and load `.env`. We also define a one-line knowledge base the agent will ground on. In production this is your real corpus (policies, manuals, tickets); one line keeps the demo self-contained.

In [1]:
import contextlib
import logging
import os

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from typing import Annotated
from pydantic import Field

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Quiet a benign framework warning about the hosted file-search tool; the tool runs fine.
logging.getLogger("agent_framework").setLevel(logging.ERROR)


def required_env(name: str, hint: str) -> str:
    """Read an environment variable or raise with a copy-pasteable hint.

    A missing endpoint or deployment name is the number-one reason a first run
    fails, so we fail fast with a clear message instead of a deep stack trace.
    """
    value = os.environ.get(name)
    if not value:
        raise SystemExit(f"ERROR: {name} is not set.\n  -> {hint}\n  See .env.example.")
    return value


# A tiny knowledge base the agent grounds on. bytes so we can upload it directly.
KB_FILENAME = "contoso_policy.txt"
KB_CONTENT = (
    b"Contoso support policy: customers on the Premium plan are entitled to a full "
    b"refund within 30 days of purchase. Standard plan refunds are issued as store credit only."
)

print("Imports OK. Knowledge base staged:", KB_FILENAME, f"({len(KB_CONTENT)} bytes)")

C:\github\ai103\lessons\01-choose-foundry-services\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
C:\github\ai103\lessons\01-choose-foundry-services\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


Imports OK. Knowledge base staged: contoso_policy.txt (166 bytes)


## LO1: choose a model

The agent reasons with the model deployment named in `FOUNDRY_MODEL`. The AI-103 habit is **task fit beats brand name**: a **small language model (SLM)** like Phi-4 wins on speed and cost for narrow, low-latency work, while a **large language model (LLM)** wins on hard, open-ended reasoning. Here we want real reasoning over a policy, so we pick a capable model. Because the code routes by **deployment name**, you can swap the underlying model later without touching application code. (Docs: [Choose the right AI model](https://learn.microsoft.com/azure/architecture/ai-ml/guide/choose-ai-model), [Small and large language models](https://learn.microsoft.com/azure/aks/concepts-ai-ml-language-models). The sibling `model_bakeoff.py` proves this LO1 tradeoff with a stopwatch.)

In [2]:
# LO1 -- the MODEL. Validate the endpoint early too; FoundryChatClient reads it from the env.
model = required_env("FOUNDRY_MODEL", "your chat model deployment name, e.g. gpt-5.1")
required_env("FOUNDRY_PROJECT_ENDPOINT", "https://<resource>.services.ai.azure.com/api/projects/<project>")

print(f"LO1  model : {model}  (capable LLM chosen for agent reasoning)")

LO1  model : gpt-5.1  (capable LLM chosen for agent reasoning)


## LO2: choose a service

Foundry **Agent Service** is a managed platform for building and running agents; `FoundryChatClient` is how the Agent Framework talks to it. We choose it over stateless chat completions because the work ahead needs **tools and state**, not one-shot text. Auth is keyless: `AzureCliCredential` binds to your `az login` identity directly. (Docs: [What is Foundry Agent Service](https://learn.microsoft.com/azure/foundry/agents/overview).)

In [3]:
# LO2 -- the SERVICE. One authenticated client for the whole notebook (keyless, Entra ID).
# Idempotent: if this cell already ran, close the prior client's underlying HTTP session first
# so a re-run does not leak connections. "client" exists only on a re-run, hence the dir() guard.
# FoundryChatClient wraps an AsyncOpenAI client at .client, which is what holds the session.
if "client" in dir():
    with contextlib.suppress(Exception):
        await client.client.close()

client = FoundryChatClient(credential=AzureCliCredential())
print("LO2  service : Microsoft Foundry Agent Service (Agent Framework), keyless auth")

LO2  service : Microsoft Foundry Agent Service (Agent Framework), keyless auth


## LO3: choose retrieval

**File search** is Foundry's built-in **retrieval-augmented generation (RAG)** tool: RAG means the model answers from your documents instead of only its training data. We upload the policy file, index it into a managed **vector store** (a database that finds text by meaning, not just keywords), and expose it as a tool. Foundry handles the parse, chunk, and embed steps for us. This is the async cell; it uploads and polls until the vector store is ready. (Docs: [File search tool for agents](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/file-search).)

In [4]:
# LO3 -- RETRIEVAL. Upload the file, index it into a managed vector store, expose file-search.
# Jupyter runs an event loop already, so we can 'await' directly at the top of a cell.
#
# Idempotent by design: re-running must not pile up duplicate vector stores and files.
# We sweep any leftovers matched by the stable name "m01-kb" and the KB filename, then
# create fresh -- guaranteeing a single file and a single vector store.
VSTORE_NAME = "m01-kb"

# Collect each list fully before deleting: deleting mid-iteration invalidates the paging cursor.
stale_vstores = [vs async for vs in client.client.vector_stores.list() if vs.name == VSTORE_NAME]
for vs in stale_vstores:
    await client.client.vector_stores.delete(vector_store_id=vs.id)

stale_files = [f async for f in client.client.files.list() if f.filename == KB_FILENAME]
for f in stale_files:
    await client.client.files.delete(file_id=f.id)

file = await client.client.files.create(file=(KB_FILENAME, KB_CONTENT), purpose="assistants")
vstore = await client.client.vector_stores.create(
    name=VSTORE_NAME, expires_after={"anchor": "last_active_at", "days": 1}
)
# create_and_poll returns only after ingestion (parse, chunk, embed) finishes -- the file is searchable.
await client.client.vector_stores.files.create_and_poll(vector_store_id=vstore.id, file_id=file.id)
file_search = client.get_file_search_tool(vector_store_ids=[vstore.id])

print("LO3  retrieval : file-search (RAG) over a managed vector store (swept then recreated)")
print(f"     swept {len(stale_vstores)} old vector store(s), {len(stale_files)} old file(s)")
print(f"     file_id={file.id}  vector_store_id={vstore.id}")

LO3  retrieval : file-search (RAG) over a managed vector store (swept then recreated)
     swept 0 old vector store(s), 0 old file(s)
     file_id=assistant-5QbWcqBgwKXS7VS2qDDcD7  vector_store_id=vs_hD1LePXx46WOYDgbEMDVffbv


## LO4: memory + tools + knowledge (with a guardrail)

The final decision assembles the agent from four parts. First the **tool**: a custom Python function the agent can call to take an action (here, open a refund ticket). The `approval_mode` is the **guardrail**: `never_require` lets the agent act on its own so the flow stays uninterrupted; use `always_require` in production so a human approves any sensitive action before it runs.

In [5]:
# LO4 -- a custom TOOL, with an approval gate (the guardrail from the slides).
@tool(approval_mode="never_require")
def open_refund_ticket(
    customer: Annotated[str, Field(description="Customer name.")],
    amount_usd: Annotated[float, Field(description="Refund amount in US dollars.")],
) -> str:
    """Open a refund ticket for a customer and return the ticket id."""
    # A real tool would call your ticketing API here.
    return f"Ticket RF-{abs(hash(customer)) % 10000:04d} opened for {customer}: ${amount_usd:.2f} refund queued."


print("LO4  tool : open_refund_ticket registered (approval_mode=never_require for the demo)")

LO4  tool : open_refund_ticket registered (approval_mode=never_require for the demo)


Now assemble the agent from **knowledge** (the file-search tool), **tools** (our function), and instructions. The agent's short-term **memory** is a **session**, which the Agent Framework uses to keep the conversation so the agent recalls earlier turns. We open that session fresh in the turn-1 cell below, so re-running the conversation always starts clean. (Docs: [Memory in Foundry Agent Service](https://learn.microsoft.com/azure/foundry/agents/concepts/what-is-memory).)

In [6]:
# LO4 -- assemble the agent from MEMORY + TOOLS + KNOWLEDGE.
# Re-running this cell just rebuilds the agent object cleanly (no server-side resource to leak).
# The session (memory) is created fresh in the turn-1 cell below, so the conversation always
# starts clean on a re-run.
agent = Agent(
    client=client,
    instructions=(
        "You are Contoso's support agent. Ground every policy answer in the refund "
        "policy via file search, and open a refund ticket with your tool when a refund applies."
    ),
    tools=[file_search, open_refund_ticket],
)

print("LO4  integration : session (memory) + custom tool + knowledge source + approval gate")

LO4  integration : session (memory) + custom tool + knowledge source + approval gate


## Run it: turn 1 forces retrieval and the tool

The customer's request forces the agent to **read the policy (retrieval)** to check eligibility, then **open a refund ticket (tool)**. Watch for both the grounded answer and the ticket id.

In [7]:
# Turn 1: forces RETRIEVAL (read the policy) and the TOOL (open the ticket).
# Idempotent: start from a fresh session so re-running the conversation is always a clean
# two-turn demo, not an ever-growing transcript.
session = agent.create_session()

q1 = ("I'm Dana Lee, a Premium customer. I bought 22 days ago for $80 and want a "
      "refund. Do I qualify, and if so please process it.")
print(f"User:  {q1}\n")
print(f"Agent: {await agent.run(q1, session=session)}")

User:  I'm Dana Lee, a Premium customer. I bought 22 days ago for $80 and want a refund. Do I qualify, and if so please process it.



Agent: You do qualify for a refund.

Per Contoso’s support policy, Premium plan customers are entitled to a full refund within 30 days of purchase . You purchased 22 days ago, which is within that window, and your plan is Premium, so you’re eligible for a full $80 refund.

I’ve opened a refund ticket and queued your refund:

- Ticket ID: RF-5616  
- Customer: Dana Lee  
- Amount: $80.00 (full refund)

You should see the refund processed to your original payment method shortly, depending on your bank or card issuer’s timelines.


## Turn 2 forces memory

This turn omits the customer's name and the amount. The agent must **recall them from the session** to answer, which proves the memory ingredient of LO4.

In [8]:
# Turn 2: forces MEMORY -- name and amount are not repeated; the agent recalls them from the session.
q2 = "Remind me which plan I'm on and what refund you just queued for me."
print(f"User:  {q2}\n")
print(f"Agent: {await agent.run(q2, session=session)}")

User:  Remind me which plan I'm on and what refund you just queued for me.



Agent: You’re on the **Premium** plan, and I’ve queued a **full refund of $80.00** for you under ticket ID **RF-5616**.


## Recap and cleanup

One agent, four decisions: a **model** (LO1), the Foundry **Agent Service** (LO2), grounded **retrieval** (LO3), and **memory + tool + knowledge** integration (LO4). The last cell deletes the managed vector store and file so you do not leave demo resources behind.

In [9]:
# Tidy up so the demo leaves nothing behind: delete every m01-kb vector store and KB
# file, and is safe to re-run even when nothing is left.
async def _sweep_demo_resources():
    # Collect each list fully before deleting: deleting mid-iteration invalidates the paging cursor.
    stale_vstores = [vs async for vs in client.client.vector_stores.list() if vs.name == "m01-kb"]
    for vs in stale_vstores:
        with contextlib.suppress(Exception):
            await client.client.vector_stores.delete(vector_store_id=vs.id)
    stale_files = [f async for f in client.client.files.list() if f.filename == KB_FILENAME]
    for f in stale_files:
        with contextlib.suppress(Exception):
            await client.client.files.delete(file_id=f.id)
    return len(stale_vstores), len(stale_files)

if "client" in dir():
    n_vs, n_files = await _sweep_demo_resources()
    print(f"Cleanup done. Deleted {n_vs} m01-kb vector store(s) and {n_files} KB file(s).")
else:
    print("Nothing to clean up (client was never created in this kernel).")

Cleanup done. Deleted 1 m01-kb vector store(s) and 1 KB file(s).
